# 05. Валидация модели: различающая способность, точность, стабильность

ТЗ §2.4. Внутренние рейтинговые модели проверяются по **способности различать заёмщиков, точности и стабильности** — это и измеряем на **отложенной тестовой выборке** (она не участвовала ни в обучении, ни в подборе гиперпараметров, ни в калибровке, ни в выборе границ рейтингов).

Что считаем: Gini/ROC-AUC, KS, Brier, ECE + calibration curve, PSI (train↔test, по годам, по признакам), стабильность по отраслям/размеру/продукту, таблица рейтингов (с доверительными интервалами и проверкой монотонности), важность признаков (скор-карта, permutation).

> ⚠️ Модель обучена на **синтетике**. Метрики характеризуют качество методологии на искусственных данных и **не переносятся** на реальный портфель.

In [1]:
import sys, json, warnings, joblib
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "risk-scoring")); sys.path.insert(0, str(ROOT))
from riskscore.features import compute_features, FEATURE_META, FEATURE_COLUMNS, NUM_FEATURES, CAT_FEATURES
from riskscore import metrics as M
from sklearn.metrics import roc_curve
MODELS, IMG = ROOT / "risk-scoring/models", ROOT / "docs/ml/img"
STR = {"inn": str, "ogrn": str, "okved": str, "region_code": str}
raw = pd.read_csv(ROOT / "data/generated/applications_raw.csv", dtype=STR)
split = pd.read_csv(ROOT / "data/generated/split.csv").split.to_numpy()
y = raw.default_flag.astype(int).to_numpy()
tr, va, te = (np.where(split == s)[0] for s in ("train", "valid", "test"))
lr_model = joblib.load(MODELS / "risk_model_woe_logit_v1.joblib")
p_lr = lr_model.predict_pd(raw)
p_lr_raw = lr_model.raw_score(raw)

# стиль графиков: спокойная палитра (проверенный набор), тонкие марки, приглушённая сетка, подписи прямо на графике
BLUE, ORANGE, AQUA, INK, MUTED, GRID, SURF = "#2a78d6", "#eb6834", "#1baf7a", "#0b0b0b", "#52514e", "#e3e2de", "#fcfcfb"
plt.rcParams.update({"figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF, "axes.edgecolor": MUTED,
                     "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK, "axes.titlecolor": INK,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID, "grid.linewidth": .7,
                     "axes.axisbelow": True, "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "semibold", "lines.linewidth": 2})
print("модель:", lr_model.version)

модель: v1.0.0-woe-logit


## 1. Сводные метрики: train / valid / test

In [2]:
rows = []
for nm, p in (("WOE-LR + скор-карта (основная)", p_lr),):
    for s, ix in (("train", tr), ("valid", va), ("test", te)):
        rows.append({**M.summary(y[ix], p[ix], nm), "sample": s})
tab = pd.DataFrame(rows)
print(tab[["name", "sample", "n", "default_rate", "auc", "gini", "ks", "brier", "ece", "mean_pd"]].round(4).to_string(index=False))
rng = np.random.default_rng(42); boots = []
for _ in range(1000):
    i = rng.integers(0, len(te), len(te)); yy = y[te][i]
    if yy.min() != yy.max(): boots.append((M.gini(yy, p_lr[te][i]), M.ks(yy, p_lr[te][i]), M.brier(yy, p_lr[te][i])))
ci = np.percentile(np.array(boots), [2.5, 97.5], axis=0)
print(f"\nTEST, основная модель: Gini {M.gini(y[te], p_lr[te]):.3f} [{ci[0,0]:.3f}; {ci[1,0]:.3f}] | KS {M.ks(y[te], p_lr[te]):.3f} [{ci[0,1]:.3f}; {ci[1,1]:.3f}] | Brier {M.brier(y[te], p_lr[te]):.4f} [{ci[0,2]:.4f}; {ci[1,2]:.4f}]")
base_brier = M.brier(y[te], np.full(len(te), y[tr].mean()))
print(f"Brier наивного прогноза «всем средний PD» = {base_brier:.4f} → Brier Skill Score = {1 - M.brier(y[te], p_lr[te]) / base_brier:.3f}")
print("требование ТЗ Gini ≥ 0.45:", "выполнено" if M.gini(y[te], p_lr[te]) >= 0.45 else "НЕ выполнено")
print(f"разрыв train→test по Gini (переобучение): {M.gini(y[tr], p_lr[tr]) - M.gini(y[te], p_lr[te]):+.3f}")

                          name sample    n  default_rate    auc   gini     ks  brier    ece  mean_pd
WOE-LR + скор-карта (основная)  train 7200        0.1024 0.8247 0.6495 0.5147 0.0738 0.0091   0.1018
WOE-LR + скор-карта (основная)  valid 2400        0.1025 0.8254 0.6509 0.5457 0.0747 0.0091   0.1025
WOE-LR + скор-карта (основная)   test 2400        0.1025 0.8297 0.6593 0.5167 0.0754 0.0115   0.1027



TEST, основная модель: Gini 0.659 [0.605; 0.711] | KS 0.517 [0.479; 0.577] | Brier 0.0754 [0.0677; 0.0832]
Brier наивного прогноза «всем средний PD» = 0.0920 → Brier Skill Score = 0.181
требование ТЗ Gini ≥ 0.45: выполнено
разрыв train→test по Gini (переобучение): -0.010


## 2. ROC и KS на тесте

In [3]:
fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.4))
for nm, p, col in (("WOE-LR (основная)", p_lr, BLUE),):
    f, t, _ = roc_curve(y[te], p[te]); ax[0].plot(f, t, color=col, label=f"{nm}: AUC {1/2*(M.gini(y[te], p[te])+1):.3f}")
ax[0].plot([0, 1], [0, 1], color=MUTED, lw=1, ls="--"); ax[0].set_xlabel("доля ложных тревог (FPR)"); ax[0].set_ylabel("доля пойманных дефолтов (TPR)")
ax[0].set_title("ROC-кривая, test"); ax[0].legend(frameon=False, loc="lower right")
f, t, th = roc_curve(y[te], p_lr[te]); k = int(np.argmax(t - f))
ax[1].plot(th[1:], t[1:], color=ORANGE, label="дефолтные (кумулятивно)"); ax[1].plot(th[1:], f[1:], color=BLUE, label="недефолтные (кумулятивно)")
ax[1].vlines(th[k], f[k], t[k], color=INK, lw=1.2); ax[1].text(th[k], (f[k] + t[k]) / 2, f"  KS = {t[k]-f[k]:.2f}\n  при PD ≈ {th[k]:.1%}", va="center", fontsize=9)
ax[1].set_xscale("log"); ax[1].set_xlabel("порог PD (лог. шкала)"); ax[1].set_ylabel("доля клиентов с PD ≥ порога"); ax[1].set_title("KS-статистика, test"); ax[1].legend(frameon=False, loc="upper right")
plt.tight_layout(); plt.savefig(IMG / "05_roc_ks.png", dpi=130); plt.close(); print("ok")

ok


## 3. Калибровка: calibration curve до и после

In [4]:
fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.6))
for a_, (ttl, p) in zip(ax, (("до калибровки (сырой скор LR)", p_lr_raw), ("после калибровки Платта (итоговый PD)", p_lr))):
    c = M.calibration_table(y[te], p[te], 10)
    a_.plot([0, .5], [0, .5], color=MUTED, lw=1, ls="--"); a_.plot(c.mean_pd, c.observed, "o-", color=BLUE, ms=6, mfc=SURF, mew=2)
    a_.set_xlabel("средний прогноз PD в децили"); a_.set_ylabel("фактическая доля дефолтов"); a_.set_title(f"{ttl}\nBrier {M.brier(y[te], p[te]):.4f}, ECE {M.ece(y[te], p[te]):.4f}", fontsize=10)
    a_.set_xlim(0, .5); a_.set_ylim(0, .5)
plt.tight_layout(); plt.savefig(IMG / "05_calibration.png", dpi=130); plt.close()
cal_tab = M.calibration_table(y[te], p_lr[te], 10); print(cal_tab.round(4).to_string(index=False))
print("средний PD на test:", round(p_lr[te].mean(), 4), "| факт:", round(y[te].mean(), 4))

  n  mean_pd  observed
240   0.0146    0.0125
240   0.0201    0.0083
240   0.0248    0.0167
240   0.0300    0.0250
240   0.0382    0.0500
240   0.0520    0.0417
240   0.0743    0.0917
240   0.1183    0.1458
240   0.2154    0.2042
240   0.4395    0.4292
средний PD на test: 0.1027 | факт: 0.1025


## 4. Таблица рейтингов на тесте (монотонность + доверительные интервалы Уилсона)

In [5]:
scale = json.load(open(MODELS / "rating_scale_v1.json", encoding="utf-8"))["grades"]
GR = [g["grade"] for g in scale]; UP = [g["pd_to"] for g in scale]
grade = np.array(GR)[np.searchsorted(UP, p_lr, side="right").clip(0, len(GR) - 1)]
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n); h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (c - h) / d, (c + h) / d
def rt(ix):
    d = pd.DataFrame({"g": grade[ix], "pd": p_lr[ix], "y": y[ix]})
    t = d.groupby("g").agg(n=("y", "size"), defaults=("y", "sum"), mean_pd=("pd", "mean"), observed=("y", "mean")).reindex(GR)
    t.insert(0, "share", t.n / t.n.sum())
    t["ci_lo"], t["ci_hi"] = zip(*[wilson(k, n) for k, n in zip(t.defaults, t.n)]); return t
rt_te = rt(te); rt_all = rt(np.arange(len(y)))
print(rt_te.round(4).to_string())
mono_te = bool(rt_te.observed.is_monotonic_increasing); mono_all = bool(rt_all.observed.is_monotonic_increasing)
print("\nмонотонность фактической доли дефолтов: test", mono_te, "| вся выборка (12 000)", mono_all)
print("доля градаций, где средний PD внутри 95% ДИ факта (test):", round(((rt_te.mean_pd >= rt_te.ci_lo) & (rt_te.mean_pd <= rt_te.ci_hi)).mean(), 2))
fig, ax = plt.subplots(figsize=(9.5, 4.3)); x = np.arange(len(GR))
ax.bar(x, rt_te.observed * 100, .58, color=BLUE, label="факт (test), 95% ДИ"); ax.errorbar(x, rt_te.observed * 100, yerr=[(rt_te.observed - rt_te.ci_lo) * 100, (rt_te.ci_hi - rt_te.observed) * 100], fmt="none", ecolor=INK, lw=1, capsize=3)
ax.plot(x, rt_te.mean_pd * 100, "o", color=ORANGE, ms=7, mfc=SURF, mew=2, label="средний PD модели")
ax.set_xticks(x); ax.set_xticklabels(GR); ax.set_yscale("log"); ax.set_ylabel("%, лог. шкала"); ax.set_title("Рейтинговые градации на отложенной выборке"); ax.legend(frameon=False, loc="upper left")
plt.tight_layout(); plt.savefig(IMG / "05_rating_table.png", dpi=130); plt.close()

      share    n  defaults  mean_pd  observed   ci_lo   ci_hi
g                                                            
AAA  0.1467  352         4   0.0159    0.0114  0.0044  0.0289
AA   0.2054  493         9   0.0248    0.0183  0.0096  0.0343
A    0.1562  375        14   0.0364    0.0373  0.0224  0.0617
BBB  0.1092  262        13   0.0544    0.0496  0.0292  0.0830
BB   0.0825  198        19   0.0767    0.0960  0.0623  0.1450
B    0.0629  151        18   0.1056    0.1192  0.0767  0.1806
CCC  0.0562  135        23   0.1497    0.1704  0.1163  0.2427
CC   0.0812  195        43   0.2267    0.2205  0.1680  0.2838
C    0.0533  128        42   0.3390    0.3281  0.2528  0.4134
D    0.0462  111        61   0.5568    0.5495  0.4569  0.6389

монотонность фактической доли дефолтов: test True | вся выборка (12 000) True
доля градаций, где средний PD внутри 95% ДИ факта (test): 1.0


## 5. Стабильность I: PSI train ↔ test, по годам заявок, по признакам
Правило: PSI < 0.1 — стабильно, 0.1–0.25 — умеренный сдвиг, > 0.25 — значимый (нужен пересмотр).

In [6]:
psi_score = M.psi(p_lr[tr], p_lr[te]); psi_valid = M.psi(p_lr[tr], p_lr[va])
print(f"PSI скора train→test: {psi_score:.4f} | train→valid: {psi_valid:.4f}")
year = pd.to_datetime(raw.application_date).dt.year.to_numpy()
psi_year = {int(yv): M.psi(p_lr[tr], p_lr[np.where(year == yv)[0]]) for yv in sorted(set(year))}
print("PSI скора train→каждый год заявок:", {k: round(v, 4) for k, v in psi_year.items()})
Xall = compute_features(raw); lr_clip = lr_model.pipeline.named_steps["clip"]; Xc = lr_clip.transform(Xall)
psi_feat = pd.Series({c: M.psi(Xc[c].iloc[tr].dropna(), Xc[c].iloc[te].dropna()) for c in NUM_FEATURES}).sort_values(ascending=False)
print("\nPSI признаков train→test (топ-6):"); print(psi_feat.head(6).round(4).to_string())
print("признаков с PSI > 0.1:", int((psi_feat > .1).sum()), "из", len(psi_feat), "| макс.", round(psi_feat.max(), 4))

PSI скора train→test: 0.0066 | train→valid: 0.0012
PSI скора train→каждый год заявок: {2023: 0.0015, 2024: 0.0029, 2025: 0.0028}



PSI признаков train→test (топ-6):
counterparties_log    0.0110
employees_log         0.0093
debt_to_revenue       0.0085
history_months        0.0079
turnover_cv           0.0077
company_age_months    0.0073
признаков с PSI > 0.1: 0 из 51 | макс. 0.011


## 6. Стабильность II: качество по сегментам (test) и по годам заявок (вся выборка вне обучения: valid+test)
Gini по сегменту считается, если в нём ≥ 150 заявок и ≥ 10 дефолтов (иначе оценка шумная).

In [7]:
d = raw.copy(); d["p"] = p_lr; d["y"] = y
d["size"] = np.where(d.employees <= 15, "микро (≤15 чел.)", "малый (>15 чел.)")
d["year"] = year
segs = []
def seg_stats(name, col, ix):
    for v, g in d.iloc[ix].groupby(col):
        r = dict(segment=name, value=str(v), n=len(g), default_rate=g.y.mean(), mean_pd=g.p.mean())
        r["gini"] = M.gini(g.y, g.p) if len(g) >= 150 and g.y.sum() >= 10 else np.nan
        segs.append(r)
for nm, col in (("отрасль", "industry"), ("форма", "legal_form"), ("размер", "size"), ("продукт", "product_code"), ("уровень региона", "region_tier")):
    seg_stats(nm, col, te)
seg_stats("год заявки (valid+test)", "year", np.concatenate([va, te]))
segdf = pd.DataFrame(segs)
print(segdf.round(3).to_string(index=False))
gs = segdf.gini.dropna()
print(f"\nGini по сегментам: мин {gs.min():.3f}, макс {gs.max():.3f}, разброс {gs.max()-gs.min():.3f}; сегментов с Gini ≥ 0.45: {(gs >= 0.45).sum()} из {len(gs)}")
print("перекалибровка по сегментам: макс. |mean_pd − факт| =", round((segdf.mean_pd - segdf.default_rate).abs().max(), 3), "(сегменты малы, шум высок)")

                segment            value    n  default_rate  mean_pd  gini
                отрасль      agriculture  124         0.097    0.069   NaN
                отрасль         catering  228         0.140    0.129 0.640
                отрасль     construction  262         0.130    0.122 0.540
                отрасль       health_edu  133         0.068    0.066   NaN
                отрасль      it_services  226         0.049    0.041 0.756
                отрасль    manufacturing  265         0.049    0.066 0.538
                отрасль    personal_serv  206         0.053    0.062 0.761
                отрасль      real_estate   68         0.015    0.028   NaN
                отрасль      retail_food  379         0.172    0.168 0.593
                отрасль        transport  210         0.029    0.060   NaN
                отрасль        wholesale  299         0.174    0.168 0.533
                  форма               IP  865         0.110    0.109 0.689
                  форма  

## 7. Важность признаков: вес в скор-карте, permutation importance на тесте
Permutation: перемешиваем один признак в test и смотрим падение Gini итоговой модели (метод не зависит от структуры модели и ловит эффекты через корреляции).

In [8]:
from sklearn.pipeline import Pipeline
tail = lr_model.pipeline[2:]                       # WOE -> отбор -> логрегрессия
def gini_from_Xc(Xc_): return M.gini(y[te], lr_model.calibrate(tail.predict_proba(Xc_)[:, 1]))
Xc_te = Xc.iloc[te].reset_index(drop=True); g0 = gini_from_Xc(Xc_te)
rng = np.random.default_rng(0); perm = {}
for c in FEATURE_COLUMNS:
    drops = []
    for _ in range(5):
        Z = Xc_te.copy(); Z[c] = Z[c].sample(frac=1, random_state=int(rng.integers(1e9))).to_numpy(); drops.append(g0 - gini_from_Xc(Z))
    perm[c] = np.mean(drops)
perm = pd.Series(perm).sort_values(ascending=False)
card = json.load(open(MODELS / "scorecard_v1.json", encoding="utf-8"))["features"]
rng_pts = pd.Series({c: max(v["points"]) - min(v["points"]) for c, v in card.items()}).sort_values(ascending=False)
print("Permutation importance (падение Gini на test), топ-12:"); print(perm.head(12).round(4).to_string())
print("\nПризнаки вне скор-карты имеют нулевую важность (не входят в модель):", int((perm.abs() < 1e-9).sum()), "из", len(perm))
fig, ax = plt.subplots(1, 2, figsize=(12.5, 5.2))
top = perm.head(12)[::-1]; ax[0].barh([FEATURE_META[c]["label"] for c in top.index], top.values, color=BLUE, height=.6); ax[0].set_title("Permutation importance (падение Gini, test)"); ax[0].set_xlabel("ΔGini")
top2 = rng_pts.head(12)[::-1]; ax[1].barh([FEATURE_META[c]["label"] for c in top2.index], top2.values, color=AQUA, height=.6); ax[1].set_title("Размах баллов признака в скор-карте"); ax[1].set_xlabel("баллы")
plt.tight_layout(); plt.savefig(IMG / "05_importance.png", dpi=130); plt.close()

Permutation importance (падение Gini на test), топ-12:
payment_burden         0.0578
loss_quarters          0.0433
equity_ratio           0.0134
company_age_months     0.0071
gross_margin           0.0058
overdue_30_cnt         0.0041
negative_equity        0.0024
revenue_cv             0.0024
inquiries_3m           0.0020
inquiries_12m          0.0020
balance_to_turnover    0.0018
account_blocked        0.0011

Признаки вне скор-карты имеют нулевую важность (не входят в модель): 39 из 64


## 8. Проверка на «здравый смысл»: PD вырастает с ростом ключевых рисков (модель не противоречит бизнес-логике)
Берём реальные заявки test и искусственно ухудшаем один фактор — средний PD не должен падать.

In [9]:
base = raw.iloc[te].copy(); pd0 = lr_model.predict_pd(base).mean()
def shift(df, **kw):
    z = df.copy()
    for k, f in kw.items(): z[k] = f(z[k])
    return z
tests = {
    "сумма заявки ×2": shift(base, requested_amount=lambda s: s * 2),
    "срок короче вдвое (платёж выше)": shift(base, term_months=lambda s: np.maximum(s // 2, 3)),
    "выручка последнего кв. −30%": shift(base, revenue_q8=lambda s: s * .7, ebitda_q8=lambda s: s * .5),
    "просрочка 90+ (150 дн.)": shift(base[base.bki_has_history == 1], bki_overdue_90_cnt=lambda s: 1, bki_overdue_60_cnt=lambda s: 1, bki_overdue_30_cnt=lambda s: 2, bki_max_overdue_days=lambda s: 150),
    "налоговый долг = 1 мес. выручки": shift(base, fns_tax_debt=lambda s: base[[f"revenue_q{k}" for k in range(5, 9)]].sum(axis=1) / 12),
    "блокировка счёта": shift(base, fns_account_blocked=lambda s: 1),
    "+3 запроса в БКИ за 3 мес.": shift(base, bki_inquiries_3m=lambda s: s + 3),
}
sens = {}
for k, z in tests.items():
    sens[k] = lr_model.predict_pd(z).mean() - lr_model.predict_pd(base.loc[z.index]).mean()
    print(f"{k:38s} Δ средний PD = {sens[k]:+.4f}  → {'OK' if sens[k] >= -1e-9 else 'ПРОТИВОРЕЧИТ'}")
print("\nисходный средний PD test:", round(pd0, 4))

сумма заявки ×2                        Δ средний PD = +0.0076  → OK
срок короче вдвое (платёж выше)        Δ средний PD = +0.0066  → OK


выручка последнего кв. −30%            Δ средний PD = +0.0131  → OK
просрочка 90+ (150 дн.)                Δ средний PD = +0.0805  → OK


налоговый долг = 1 мес. выручки        Δ средний PD = +0.0011  → OK
блокировка счёта                       Δ средний PD = +0.0049  → OK


+3 запроса в БКИ за 3 мес.             Δ средний PD = +0.0129  → OK

исходный средний PD test: 0.1027


## 9. Сохранение всех метрик (для отчёта)

In [10]:
out = dict(
    model=lr_model.version, dataset="synth-v1.0", seed=42,
    sizes=dict(train=len(tr), valid=len(va), test=len(te)), default_rate=dict(train=float(y[tr].mean()), valid=float(y[va].mean()), test=float(y[te].mean())),
    metrics={f"{r['name']}|{r['sample']}": {k: r[k] for k in ("auc", "gini", "ks", "brier", "ece", "mean_pd", "n")} for r in rows},
    test_ci=dict(gini=[float(ci[0, 0]), float(ci[1, 0])], ks=[float(ci[0, 1]), float(ci[1, 1])], brier=[float(ci[0, 2]), float(ci[1, 2])]),
    brier_skill=float(1 - M.brier(y[te], p_lr[te]) / base_brier),
    psi=dict(score_train_test=psi_score, score_train_valid=psi_valid, by_year=psi_year, features_max=float(psi_feat.max()), features_gt_0_1=int((psi_feat > .1).sum()), top=psi_feat.head(5).to_dict()),
    rating_test=rt_te.reset_index().rename(columns={"g": "grade", "index": "grade"}).to_dict("records"), rating_monotone=dict(test=mono_te, all=mono_all),
    rating_all=rt_all.reset_index().rename(columns={"g": "grade", "index": "grade"}).to_dict("records"),
    segments=segdf.where(segdf.notna(), None).to_dict("records"),
    permutation_top=perm.head(12).to_dict(),
    sensitivity=sens, calibration_test=cal_tab.to_dict("records"), gini_gap_train_test=float(M.gini(y[tr], p_lr[tr]) - M.gini(y[te], p_lr[te])),
)
json.dump(out, open(ROOT / "docs/ml/metrics.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1, default=float)
print("записано docs/ml/metrics.json;", len(json.dumps(out, default=float)) // 1000, "КБ")

записано docs/ml/metrics.json; 12 КБ
